# Phishing URL Detection on the PhiUSIIL Dataset: Probabilistic & Bayesian Modeling

**Dataset:** [PhiUSIIL Phishing URL (Website) Dataset](https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset) (UCI Machine Learning Repository / Kaggle)  
**Instances:** 235,795 URLs (`134,850` legitimate [`label = 1`], `100,945` phishing [`label = 0`])  

### Notebook Pipeline
1. **Environment Setup & Data Acquisition** — Load the dataset reproducibly from local disk or via `kagglehub`.
2. **Exploratory Data Analysis (EDA) & Data Hygiene** — Inspect schema, class balance, missing values, and filter non-predictive high-cardinality identifier columns (`FILENAME`, `URL`, `Title`, `Domain`, `TLD`).
3. **Model 1: Gaussian Naïve Bayes Baseline** — Standardized features (`StandardScaler`) evaluated with 10-fold stratified cross-validation.
4. **Model 2: Bayesian Belief Network (`pgmpy`)** — Feature discretization, structure learning with `HillClimbSearch` (`BicScore`), parameter estimation via `BayesianEstimator` (`BDeu`), and exact probabilistic inference using `VariableElimination`.
5. **Model 3: Bayesian Logistic Regression (`pymc` + `arviz`)** — Full posterior estimation and uncertainty quantification over top predictive webpage/URL features (`URLSimilarityIndex`, `HasSocialNet`, `HasCopyrightInfo`, `HasDescription`, `IsHTTPS`).
6. **Model Comparison & Analytical Takeaways** — Summary of findings and leakage considerations (`URLSimilarityIndex`).

## 1. Environment Setup & Configuration

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Probabilistic Graphical Models
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore, BayesianEstimator
from pgmpy.inference import VariableElimination

# Bayesian Generalized Linear Modeling
import pymc as pm
import arviz as az

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment initialized successfully.")

## 2. Data Acquisition & Loading

Place `PhiUSIIL_Phishing_URL_Dataset.csv` in the project root (or `data/`), or allow the loader to fetch it automatically via `kagglehub`.

In [ ]:
def load_phiusiil_dataset() -> pd.DataFrame:
    candidate_paths = [
        Path("PhiUSIIL_Phishing_URL_Dataset.csv"),
        Path("data/PhiUSIIL_Phishing_URL_Dataset.csv"),
        Path("../data/PhiUSIIL_Phishing_URL_Dataset.csv"),
    ]
    for path in candidate_paths:
        if path.exists():
            print(f"Loading dataset from local path: {path.resolve()}")
            return pd.read_csv(path)

    print("Local CSV not found. Downloading via kagglehub...")
    import kagglehub
    dataset_dir = Path(kagglehub.dataset_download("ndarvind/phiusiil-phishing-url-dataset"))
    csv_files = list(dataset_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {dataset_dir}")
    print(f"Loading dataset from: {csv_files[0]}")
    return pd.read_csv(csv_files[0])


df = load_phiusiil_dataset()
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 3. Exploratory Data Analysis & Preprocessing

In [ ]:
# Check missing values and target distribution
missing_total = df.isnull().sum().sum()
print(f"Total missing values: {missing_total}")

class_counts = df["label"].value_counts()
class_props = df["label"].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({
    "Class": ["Legitimate (1)", "Phishing (0)"],
    "Count": [class_counts.get(1, 0), class_counts.get(0, 0)],
    "Percentage (%)": [round(class_props.get(1, 0), 2), round(class_props.get(0, 0), 2)]
})
display(summary_df)

# Visualize class distribution and key feature distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df, x="label", ax=axes[0], palette=["#d95f02", "#1b9e77"])
axes[0].set_title("Target Class Distribution (0 = Phishing, 1 = Legitimate)")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")

sns.boxplot(data=df, x="label", y="URLSimilarityIndex", ax=axes[1], palette=["#d95f02", "#1b9e77"])
axes[1].set_title("URLSimilarityIndex by Class")

sns.barplot(data=df, x="label", y="IsHTTPS", ax=axes[2], palette=["#d95f02", "#1b9e77"])
axes[2].set_title("Proportion of HTTPS by Class")

plt.tight_layout()
plt.show()

In [ ]:
# Drop high-cardinality text/identifier columns
DROP_COLS = ["FILENAME", "URL", "Title", "Domain", "TLD"]
existing_drop_cols = [c for c in DROP_COLS if c in df.columns]

df_model = df.drop(columns=existing_drop_cols).copy()
df_model.replace([np.inf, -np.inf], np.nan, inplace=True)
df_model.dropna(inplace=True)

X = df_model.drop(columns=["label"])
y = df_model["label"].astype(int)

print(f"Dropped identifier columns: {existing_drop_cols}")
print(f"Modeling feature matrix shape: {X.shape}")

# Top correlations with label
correlations = X.apply(lambda col: col.corr(y)).abs().sort_values(ascending=False)
print("\nTop 10 Features by Absolute Pearson Correlation with Label:")
display(correlations.head(10).to_frame(name="Abs_Correlation"))

## 4. Model 1: Gaussian Naïve Bayes (10-Fold Stratified Cross-Validation)

We establish a fast generative baseline using `StandardScaler` + `GaussianNB` across 10 stratified folds.

In [ ]:
nb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("gnb", GaussianNB())
])

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_results = cross_validate(nb_pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)

nb_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean(),
    ],
    "Std Dev": [
        cv_results["test_accuracy"].std(),
        cv_results["test_precision"].std(),
        cv_results["test_recall"].std(),
        cv_results["test_f1"].std(),
        cv_results["test_roc_auc"].std(),
    ]
})
display(nb_summary.round(4))

# Hold-out classification report & confusion matrix
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
nb_pipeline.fit(X_train, y_train)
y_pred = nb_pipeline.predict(X_test)
print("\nHold-Out Classification Report (Gaussian NB):")
print(classification_report(y_test, y_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Phishing (0)", "Legitimate (1)"], cmap="Blues")
plt.title("Gaussian Naïve Bayes — Hold-Out Confusion Matrix")
plt.grid(False)
plt.show()

## 5. Model 2: Bayesian Belief Network (`pgmpy`)

Discrete Bayesian Networks capture conditional dependencies among webpage security indicators. Here we:
1. Select key binary and continuous indicators and discretize `NoOfExternalRef` (`low`, `medium`, `high`) and `URLSimilarityIndex` (`low`, `medium`, `high`).
2. Learn network structure via **Hill Climb Search** using the **BIC score**.
3. Estimate Conditional Probability Distributions (CPDs) with **Bayesian Estimation (`BDeu`)**.
4. Query posterior probabilities using **Variable Elimination**.

In [ ]:
bbn_cols = [
    "IsHTTPS",
    "HasSocialNet",
    "HasCopyrightInfo",
    "HasDescription",
    "NoOfExternalRef",
    "URLSimilarityIndex",
    "label",
]
bbn_df = df_model[bbn_cols].copy()

# Discretize continuous variables into interpretable ordinal bins
bbn_df["NoOfExternalRef_bin"] = pd.qcut(
    bbn_df["NoOfExternalRef"], q=3, labels=["low", "medium", "high"], duplicates="drop"
).astype(str)

bbn_df["URLSimilarity_bin"] = pd.cut(
    bbn_df["URLSimilarityIndex"],
    bins=[-np.inf, 50, 85, np.inf],
    labels=["low", "medium", "high"]
).astype(str)

discrete_features = [
    "IsHTTPS",
    "HasSocialNet",
    "HasCopyrightInfo",
    "HasDescription",
    "NoOfExternalRef_bin",
    "URLSimilarity_bin",
    "label",
]
bbn_discrete = bbn_df[discrete_features].copy()

# Split for BBN training & evaluation
bbn_train, bbn_test = train_test_split(
    bbn_discrete, test_size=0.2, stratify=bbn_discrete["label"], random_state=RANDOM_STATE
)

# Structure learning with HillClimbSearch + BicScore
hc = HillClimbSearch(bbn_train)
best_dag = hc.estimate(scoring_method=BicScore(bbn_train))
print("Learned BBN Directed Edges:")
for edge in best_dag.edges():
    print(f"  {edge[0]} -> {edge[1]}")

# Fit CPDs with BDeu smoothing
bbn_model = BayesianNetwork(best_dag.edges())
bbn_model.fit(bbn_train, estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=10)
assert bbn_model.check_model(), "BBN model validation failed!"

# Exact inference queries
infer = VariableElimination(bbn_model)

q_high_risk = infer.query(
    variables=["label"],
    evidence={"IsHTTPS": 0, "HasSocialNet": 0, "URLSimilarity_bin": "low"}
)
print("\nPosterior P(label | IsHTTPS=0, HasSocialNet=0, URLSimilarity=low):")
print(q_high_risk)

q_low_risk = infer.query(
    variables=["label"],
    evidence={"IsHTTPS": 1, "HasSocialNet": 1, "URLSimilarity_bin": "high"}
)
print("\nPosterior P(label | IsHTTPS=1, HasSocialNet=1, URLSimilarity=high):")
print(q_low_risk)

## 6. Model 3: Bayesian Logistic Regression (`pymc` + `arviz`)

To quantify parameter uncertainty alongside predictive log-odds, we fit a Bayesian Logistic Regression model on a stratified subsample (`n = 10,000`) using weakly informative Normal priors $\beta_j \sim \mathcal{N}(0, 5)$.

In [ ]:
glm_features = [
    "URLSimilarityIndex",
    "HasSocialNet",
    "HasCopyrightInfo",
    "HasDescription",
    "IsHTTPS",
]

# Stratified subsample for fast MCMC sampling
df_sub, _ = train_test_split(
    df_model, train_size=10000, stratify=df_model["label"], random_state=RANDOM_STATE
)

scaler_glm = StandardScaler()
X_glm = scaler_glm.fit_transform(df_sub[glm_features])
y_glm = df_sub["label"].values

with pm.Model() as bayesian_logit_model:
    intercept = pm.Normal("intercept", mu=0, sigma=5)
    betas = pm.Normal("betas", mu=0, sigma=5, shape=len(glm_features))

    logits = intercept + pm.math.dot(X_glm, betas)
    p = pm.Deterministic("p", pm.math.sigmoid(logits))

    y_obs = pm.Bernoulli("y_obs", p=p, observed=y_glm)

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        target_accept=0.9,
        random_seed=RANDOM_STATE,
        progressbar=True,
    )

# Posterior summary & convergence diagnostics
posterior_summary = az.summary(trace, var_names=["intercept", "betas"], hdi_prob=0.95)
posterior_summary.index = ["intercept"] + glm_features
display(posterior_summary)

# Forest plot of posterior 95% HDIs
az.plot_forest(trace, var_names=["betas"], combined=True, hdi_prob=0.95, figsize=(9, 4))
plt.gca().set_yticklabels(glm_features[::-1])
plt.axvline(0, color="red", linestyle="--", alpha=0.7)
plt.title("Bayesian Logistic Regression: 95% Highest Density Intervals (Standardized Log-Odds)")
plt.tight_layout()
plt.show()

## 7. Key Takeaways & Methodological Notes

- **Baseline Performance & Leakage Awareness:** `GaussianNB` achieves near-ceiling cross-validation metrics on PhiUSIIL largely because engineered domain-similarity features such as `URLSimilarityIndex` strongly separate legitimate and phishing records. In production pipelines, evaluate models both **with and without** `URLSimilarityIndex` to test generalization against zero-day phishing domains.
- **Interpretable Causal/Dependency Reasoning (`pgmpy`):** The Bayesian Belief Network provides transparent conditional probability queries (`VariableElimination`) for missing or partial evidence scenarios (e.g., evaluating risk when webpage HTML tags have not yet been scraped).
- **Uncertainty Quantification (`pymc`):** Bayesian Logistic Regression supplies full posterior distributions and 95% Highest Density Intervals (HDIs) for every predictor, verifying both effect direction and MCMC convergence (`r_hat ≈ 1.00`).